# Task 02 — Automatic Instance Segmentation

**Dataset:** COCO val2017, 400-image subset with instance masks

**Protocol:** benchmark — real mask mAP metrics

**Models:** Ultralytics yolo-seg, RF-DETR-Seg nano/small/medium

In [ ]:
import json, sys, os
from pathlib import Path

# Add shared dir to path
NB_ROOT = Path(__file__).parent.parent if "__file__" in dir() else Path("/home/arash/PycharmProjects/VisionServeX/notebook")
sys.path.insert(0, str(NB_ROOT / "shared"))
os.chdir(str(NB_ROOT.parent))

from paths import COCO_400_ANN, COCO_400_IMAGES, SMOKE_IMG, SMOKE_ANN, NB_ROOT, REPO_ROOT
from display import clean, scan_text
from commands import run
from notebook_utils import write_status

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

import visionservex
print(f"VisionServeX {visionservex.__version__}")

TASK = NB_ROOT / '02_automatic_segmentation'


In [ ]:
# Load all segmentation evidence
seg_rows = []

# Ultralytics v2.27
seg_v227_path = REPO_ROOT / "reports/segmentation_auto_instance_400_v227_source.json"
if seg_v227_path.exists():
    d227 = json.loads(seg_v227_path.read_text())
    for r in d227.get("rows", []):
        seg_rows.append({
            "model_id": r["model_id"], "source_engine": r.get("source_engine","ultralytics"),
            "mask_mAP50_95": r.get("mask_mAP50_95"), "mask_AP50": r.get("mask_AP50"),
            "mask_AP75": r.get("mask_AP75"), "latency_ms_p50": r.get("latency_ms_p50"),
            "n_images": r.get("n_images",400), "n_predictions": r.get("n_predictions",0),
            "invalid_mask_count": 0, "status": "benchmark_passed"})

# RF-DETR-Seg v231 (small)
rf_v231_path = REPO_ROOT / "reports/rfdetr_segmentation_400_v231.json"
if rf_v231_path.exists():
    d231 = json.loads(rf_v231_path.read_text())
    for r in d231.get("rows",[]):
        if r["status"]=="ok":
            seg_rows.append({"model_id":r["model_id"],"source_engine":"visionservex",
                              "mask_mAP50_95":r["mask_mAP50_95"],"mask_AP50":r.get("mask_AP50"),
                              "mask_AP75":r.get("mask_AP75"),"latency_ms_p50":r.get("latency_ms_p50"),
                              "n_images":r["n_images"],"n_predictions":r["n_predictions"],
                              "invalid_mask_count":r.get("invalid_mask_count",0),"status":"benchmark_passed"})

# RF-DETR-Seg v232 (nano, medium)
rf_v232_path = REPO_ROOT / "reports/rfdetr_seg_all_sizes_v232.json"
if rf_v232_path.exists():
    d232 = json.loads(rf_v232_path.read_text())
    for r in d232.get("rows",[]):
        if r["status"]=="ok":
            seg_rows.append({"model_id":r["model_id"],"source_engine":"visionservex",
                              "mask_mAP50_95":r["mask_mAP50_95"],"mask_AP50":r.get("mask_AP50"),
                              "mask_AP75":r.get("mask_AP75"),"latency_ms_p50":r.get("latency_ms_p50"),
                              "n_images":r["n_images"],"n_predictions":r["n_predictions"],
                              "invalid_mask_count":r.get("invalid_mask_count",0),"status":"benchmark_passed"})

seg_df = pd.DataFrame(seg_rows).sort_values("mask_mAP50_95",ascending=False).reset_index(drop=True)
seg_df["rank"] = range(1,len(seg_df)+1)
seg_df.to_csv(TASK / "reports/segmentation_leaderboard.csv", index=False)
seg_df.to_json(TASK / "reports/segmentation_leaderboard.json", orient="records", indent=2)

seg_disp = seg_df[["rank","model_id","source_engine","mask_mAP50_95","mask_AP50","latency_ms_p50"]].copy()
for c in ["mask_mAP50_95","mask_AP50","latency_ms_p50"]:
    if c in seg_disp.columns:
        seg_disp[c] = seg_disp[c].apply(lambda v: "n/a" if (v!=v or v is None) else v)
print(seg_disp.to_string(index=False))


In [ ]:
from plots import horizontal_bar
import pandas as pd
raw_seg = pd.read_csv(TASK / "reports/segmentation_leaderboard.csv")
horizontal_bar(raw_seg, "mask_mAP50_95", "model_id",
               title="Auto Instance Segmentation — mask mAP50:95",
               xlabel="mask mAP50:95 (COCO val2017, 400 images)",
               color="#8b5cf6", out=TASK / "plots/mask_mAP50_95_by_model.png")
horizontal_bar(raw_seg, "mask_AP50", "model_id",
               title="Auto Segmentation — mask AP50",
               xlabel="mask AP50", color="#16a34a",
               out=TASK / "plots/mask_AP50_by_model.png")


In [ ]:
# Conclusion
if len(seg_df):
    best = seg_df.iloc[0]
    vsx = seg_df[seg_df["source_engine"]=="visionservex"]
    ult = seg_df[seg_df["source_engine"]=="ultralytics"]
    print(f"Best overall      : {best['model_id']} = {best['mask_mAP50_95']:.4f}")
    if len(vsx):
        gap = float(best["mask_mAP50_95"]) - float(vsx.iloc[0]["mask_mAP50_95"])
        print(f"Best VisionServeX : {vsx.iloc[0]['model_id']} = {vsx.iloc[0]['mask_mAP50_95']:.4f}  (gap: {gap:.4f})")
    if len(ult):
        print(f"Best Ultralytics  : {ult.iloc[0]['model_id']} = {ult.iloc[0]['mask_mAP50_95']:.4f}")
    print(f"VisionServeX still trails. rfdetr-seg-large pending.")

write_status(TASK, task="automatic_segmentation", dataset="coco_val2017_400",
             status="benchmark_passed", n_models=len(seg_df))
